#### Code to calculate VIF

In [2]:
from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF


def get_vif_ledger(df: pd.DataFrame):
    """
    Function to compute Variance Inflation Factor (VIF).
    """
    _df = sm.add_constant(df)

    # Compute VIF for each feature.
    vif_ledger = [[f_name, VIF(_df.values, idx)] for idx, f_name in enumerate(_df.columns)]

    # Create VIF dataframe and sort in descending order of VIF value.
    vif_df = (
        pd.DataFrame(vif_ledger, columns=["Feature", "VIF"])
        .sort_values(by="VIF", ascending=False)
        .reset_index(drop=True)
    )

    return vif_df

#### Code to remove Multicollinearity

In [3]:
def resolve_multicollinearity(df: pd.DataFrame, vif_threshold: int = 5, display_vif: bool = True):
    """
    Function to resolve multicollinearity in the dataset.
    """
    _df_nmc = df.copy()  # DataFrame with no multicollinearity.
    mc_resolved = False
    ftrs_removed = list()
    itr = 0

    while not mc_resolved:
        print("\n" + "-" * 20 + f" Iteration #{(itr := itr + 1)} " + "-" * 20)
        print("Calculating VIF to check Multicollinearity in dataset.")
        vif_df = get_vif_ledger(df=_df_nmc)

        if display_vif:
            print("\nVIF ledger:")
            display(vif_df)

        for idx, row in vif_df.iterrows():
            f_name, vif_value = row[["Feature", "VIF"]]
            if vif_value > vif_threshold:
                print(f"Dropping feature: '{f_name}' with VIF value: {vif_value:.2f}")
                _df_nmc = _df_nmc.drop(columns=f_name)
                ftrs_removed.append(f_name)
                break
        else:
            # If traversing VIF ledger completes without interruption,
            # then multicollinearity is resolved in the dataset.
            mc_resolved = True

        print("-" * 54 + "\n")

    print("\n" + "=" * 65)
    print("Resolved Multicollinearity in dataset by dropping below features.")
    print("Features removed:", ftrs_removed)
    print("Features retained:", _df_nmc.columns.to_list())

    # Return final DataFrame after resolving multicollinearity.
    return _df_nmc

#### Linear Regression Pipeline

In [4]:
class SimpleLinearRegression:
    """
    A Simple Linear Regression pipeline that performs:
        1. Train test split.
        2. Train a Linear regression model.
        3. Compute residuals.
        4. Print Linear Regression report.
    """

    def __init__(self, df: pd.DataFrame, target: str, random_state=42):
        """
        Simple Linear Regression pipeline.
        """
        # STEP #1: Training setup.
        self.X = df.drop(columns=target)
        self.y = df[target]

        # STEP #2: Train test split.
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.X, self.y, test_size=0.3, random_state=random_state
        )

        # STEP #3: Train Linear Regression model.
        self.lr_model = LinearRegression()
        self.lr_model.fit(self.X_train, self.y_train)

        # STEP #4: Calculate residuals.
        self.y_pred = self.lr_model.predict(self.X_test)
        self.residuals = self.y_test.values - self.y_pred

    def _adjusted_r2_score(self, _X, r2_score: float) -> float:
        """
        Function to calculate adjusted r-squared score.
        """
        r, c = _X.shape
        return 1 - ((1 - r2_score) * (r - 1) / (r - c - 1))

    def print_report(self):
        """
        Function to print detailed report.
        """
        print("\nLinear Regression Results\n" + "=" * 25)

        print("\nCoefficients\n" + "-" * 12)
        coefficients = self.lr_model.coef_.round(6).tolist()

        table = PrettyTable()
        table.field_names = ["Feature", "Coefficient"]
        table.align = {"Feature": "l", "Coefficient": "r"}
        for idx, f_name in enumerate(self.lr_model.feature_names_in_):
            table.add_row([f_name, coefficients[idx]])
        print(table)

        print("\nR-squared\n" + "-" * 9)
        tr_r2_score = self.lr_model.score(self.X_train, self.y_train)
        ts_r2_score = self.lr_model.score(self.X_test, self.y_test)
        print("Training R-squared:", round(tr_r2_score, 6))
        print("Testing R-squared:", round(ts_r2_score, 6))

        print("\nAdj. R-squared\n" + "-" * 14)
        print("Training Adj. R-squared:", round(self._adjusted_r2_score(self.X_train, tr_r2_score), 6))
        print("Testing Adj. R-squared:", round(self._adjusted_r2_score(self.X_test, ts_r2_score), 6))